In [ ]:
%run "./00_CC_Mapping_Setup.ipynb" 

In [ ]:
%sql
/* ===================================================================================
   METRIC: TP05 - TP Gov't Int
   
   DATA SOURCES:
   1. Master AU List: hive_metastore.ra_adido_2025.fy25_list_of_units
   2. TP Unit Mapping: hive_metastore.ra_adido_2025.third_party_unit_mapping
   3. Third Party Data: hive_metastore.ra_adido_2025.abac_request_paul_v3
   
   BUSINESS QUESTION: 
   Number of third party relationships or engagements which required 1. the arrangement 
   of government issued approval, license and/or permits, etc. OR 2. the third party 
   to apply or interact with a PO on behalf of TD?
   
   LOGIC SUMMARY:
   1. MASTER AU ANCHOR: Uses the Master AU list as the absolute key for the output. 
      Only AUs in this filtered list (Canada, HK, Barbados + US_OR_CANADA = 'CANADA') 
      will appear in the final report.
   2. RISK FLAG LOGIC: Flags engagements as high risk if KPI_Number = '8.2' AND 
      Value = 'YES'.
   3. TPRM STRING MAPPING: Maps flagged TP engagements to AU IDs by checking if the 
      'TPRM_Units' string from the mapping table exists inside the text of the 
      'owning_lob' or 'lob_using' columns.
      - Blocks blank mapping strings using NULLIF to prevent wildcard explosions.
      - Uses Databricks RLIKE with word boundaries (\b) for exact phrase matching.
   4. FINAL OUTPUT: Strict 6-column structure, counting the distinct numerical 
      engagements mapped to the AU and converting NULL sums to '0'.
=================================================================================== */

WITH Master_AUs AS (
    -- 1. Grab Master List data: This is our strict anchor for the final output
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

Flagged_Engagements AS (
    -- 2. Filter the TP data strictly based on KPI 8.2 and Value = YES
    SELECT DISTINCT
        EngagementNumber,
        owning_lob,
        lob_using
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3
    WHERE EngagementNumber IS NOT NULL
      AND TRIM(KPI_Number) = '8.2' 
      AND UPPER(TRIM(Value)) = 'YES'
),

Engagements_By_AU AS (
    -- 3. Map the flagged engagements using the safe RLIKE TPRM unit mapping logic
    SELECT 
        TRIM(CAST(map.Assessable_Unit_ID AS STRING)) AS Mapped_AU_ID,
        COUNT(DISTINCT f.EngagementNumber) AS Match_Count
    FROM hive_metastore.ra_adido_2025.third_party_unit_mapping map
    
    -- FIXED JOIN: Uses Regex word boundaries and completely blocks blank mapping strings
    INNER JOIN Flagged_Engagements f
        ON NULLIF(TRIM(map.TPRM_Units), '') IS NOT NULL
       AND (
           UPPER(f.owning_lob) RLIKE '\\b' || UPPER(TRIM(map.TPRM_Units)) || '\\b'
           OR UPPER(f.lob_using) RLIKE '\\b' || UPPER(TRIM(map.TPRM_Units)) || '\\b'
       )
    WHERE map.Assessable_Unit_ID IS NOT NULL
    GROUP BY TRIM(CAST(map.Assessable_Unit_ID AS STRING))
)

-- 4. Final Template: Strict 6-column output anchored to Master AU List
SELECT 
    a.BusinessID,                          
    a.AU_Name,                             
    a.Business_Segment,
    'TP05' AS QuestionID,               
    COALESCE(CAST(e.Match_Count AS STRING), '0') AS Response,
    a.In_ABAC_AU_List
    
FROM Master_AUs a
LEFT JOIN Engagements_By_AU e 
    ON a.BusinessID = e.Mapped_AU_ID
ORDER BY a.BusinessID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: TP05 Drill-Down
   
   PURPOSE: Shows EVERY third-party engagement that triggered the KPI 8.2 ('YES') rule, 
   regardless of whether the string mapped to an AU, or whether that AU exists in 
   the Master List. Useful for tracking unmapped high-risk government interactions.
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

Flagged_Engagements AS (
    SELECT DISTINCT
        EngagementNumber,
        owning_lob,
        lob_using,
        TRIM(KPI_Number) AS KPI_Number,
        TRIM(Value) AS KPI_Value,
        'KPI 8.2 Trigger' AS Risk_Trigger
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3
    WHERE EngagementNumber IS NOT NULL
      AND TRIM(KPI_Number) = '8.2' 
      AND UPPER(TRIM(Value)) = 'YES'
)

SELECT DISTINCT
    COALESCE(TRIM(CAST(map.Assessable_Unit_ID AS STRING)), 'UNMAPPED_ENGAGEMENT') AS BusinessID,
    COALESCE(mast.In_ABAC_AU_List, 'No') AS In_ABAC_AU_List,
    f.EngagementNumber,
    f.Risk_Trigger,
    f.KPI_Number,
    f.KPI_Value,
    f.owning_lob AS Raw_Col_K_owning_lob,
    f.lob_using AS Raw_Col_L_lob_using,
    map.TPRM_Units AS Matched_Mapping_String
FROM Flagged_Engagements f

-- Join to mapping table using the safe RLIKE logic
LEFT JOIN hive_metastore.ra_adido_2025.third_party_unit_mapping map
    ON NULLIF(TRIM(map.TPRM_Units), '') IS NOT NULL
   AND (
       UPPER(f.owning_lob) RLIKE '\\b' || UPPER(TRIM(map.TPRM_Units)) || '\\b'
       OR UPPER(f.lob_using) RLIKE '\\b' || UPPER(TRIM(map.TPRM_Units)) || '\\b'
   )
   
LEFT JOIN Master_AUs mast
    ON TRIM(CAST(map.Assessable_Unit_ID AS STRING)) = mast.BusinessID
ORDER BY BusinessID, f.EngagementNumber;